# Training

In [ ]:
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.datasets import DenoisingAutoEncoderDataset
from torch.utils.data import DataLoader
import random
import os

# =========================
# CONFIG
# =========================
MODEL_NAME = "all-MiniLM-L6-v2"
SAVE_DIR = "checkpoints"
SAVE_EPOCHS = {5, 10, 15, 20}   # epochs to save

EPOCHS = 20
BATCH_SIZE = 32

os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# LOAD DATA
# =========================
with open("queries.txt") as f:
    queries = [line.strip() for line in f if line.strip()]

random.shuffle(queries)

train_queries = queries[:27000]

# =========================
# MODEL
# =========================
model = SentenceTransformer(MODEL_NAME)

train_dataset = DenoisingAutoEncoderDataset(train_queries)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

train_loss = losses.DenoisingAutoEncoderLoss(
    model,
    decoder_name_or_path=MODEL_NAME
)

# =========================
# CUSTOM CALLBACK
# =========================
class EpochCheckpointCallback:
    def __init__(self, model, save_epochs, save_dir):
        self.model = model
        self.save_epochs = save_epochs
        self.save_dir = save_dir
        self.current_epoch = 0

    def __call__(self, score, epoch, steps):
        # SentenceTransformers passes epoch starting from 0
        current_epoch = epoch + 1

        if current_epoch in self.save_epochs:
            path = os.path.join(self.save_dir, f"model-epoch-{current_epoch}")
            print(f"\n💾 Saving checkpoint at epoch {current_epoch} → {path}")
            self.model.save(path)

# =========================
# TRAIN
# =========================
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    weight_decay=0,
    scheduler="constantlr",
    optimizer_params={"lr": 3e-5},
    show_progress_bar=True,
    callback=EpochCheckpointCallback(model, SAVE_EPOCHS, SAVE_DIR)
)

# save final model
model.save(os.path.join(SAVE_DIR, "final-model"))

# Evaluation

In [2]:
import json
import os
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

C:\Users\molla\anaconda3\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
# === CONFIG ===
MODEL_PATHS = [
    "model-epoch-5",
    "model-epoch-10",
    "model-epoch-15",
    "model-epoch-20"
]

EVAL_FILE = "eval_data.json"

In [4]:
# === LOAD DATA ===
with open(EVAL_FILE) as f:
    data = json.load(f)

def similarity(model, a, b):
    emb = model.encode([a, b])
    return cosine_similarity([emb[0]], [emb[1]])[0][0]

def evaluate_model(model_path):
    model = SentenceTransformer(model_path)

    total = 0
    correct = 0
    pos_scores = []
    neg_scores = []

    for item in data:
        for (a, b) in item["positive"]:
            pos_sim = similarity(model, a, b)
            pos_scores.append(pos_sim)

            neg_sims = [similarity(model, na, nb) for (na, nb) in item["negative"]]
            max_neg = max(neg_sims)
            neg_scores.append(max_neg)

            if pos_sim > max_neg:
                correct += 1

            total += 1

    accuracy = correct / total
    avg_pos = np.mean(pos_scores)
    avg_neg = np.mean(neg_scores)
    margin = avg_pos - avg_neg

    return {
        "model": model_path,
        "accuracy": accuracy,
        "avg_pos": avg_pos,
        "avg_neg": avg_neg,
        "margin": margin
    }

# === RUN EVALUATION ===
results = []

for path in MODEL_PATHS:
    print(f"Evaluating {path}...")
    res = evaluate_model(path)
    results.append(res)

# === SORT BY BEST MODEL ===
results = sorted(results, key=lambda x: (x["margin"], x["accuracy"]), reverse=True)

# === PRINT RESULTS ===
print("\n===== MODEL COMPARISON =====\n")

for r in results:
    print(f"Model: {r['model']}")
    print(f"  Accuracy : {r['accuracy']:.4f}")
    print(f"  Margin   : {r['margin']:.4f}")
    print(f"  Avg Pos  : {r['avg_pos']:.4f}")
    print(f"  Avg Neg  : {r['avg_neg']:.4f}")
    print("")

# === BEST MODEL ===
best = results[0]

print("===== BEST MODEL =====")
print(f"Model: {best['model']}")
print(f"Accuracy: {best['accuracy']:.4f}")
print(f"Margin  : {best['margin']:.4f}")
print("======================")

FileNotFoundError: [Errno 2] No such file or directory: 'eval_data.json'

# Sample Test Dataset

In [ ]:
[
  {"query":"red phone with charger","positive":[["red","phone"],["charger","phone"]],"negative":[["red","laptop"],["charger","tv"]]},
  {"query":"blue phone","positive":[["blue","phone"]],"negative":[["blue","laptop"]]},
  {"query":"black iphone 128gb","positive":[["black","iphone"],["128gb","iphone"]],"negative":[["black","tv"],["128gb","camera"]]},
  {"query":"samsung phone with charger","positive":[["samsung","phone"],["charger","phone"]],"negative":[["samsung","laptop"],["charger","tablet"]]},
  {"query":"iphone 256gb white","positive":[["256gb","iphone"],["white","iphone"]],"negative":[["256gb","tv"],["white","fan"]]},
  {"query":"cheap red smartphone","positive":[["red","smartphone"]],"negative":[["red","shoes"]]},
  {"query":"laptop with 16gb ram","positive":[["16gb","laptop"]],"negative":[["16gb","phone"]]},
  {"query":"gaming laptop 32gb ram","positive":[["32gb","laptop"]],"negative":[["32gb","phone"]]},
  {"query":"black shoes for men","positive":[["black","shoes"]],"negative":[["black","phone"]]},
  {"query":"white sneakers","positive":[["white","sneakers"]],"negative":[["white","laptop"]]},
  {"query":"phone with fast charger","positive":[["charger","phone"]],"negative":[["charger","tv"]]},
  {"query":"wireless charger for iphone","positive":[["charger","iphone"]],"negative":[["charger","laptop"]]},
  {"query":"iphone 14 black","positive":[["iphone","iphone"],["black","iphone"]],"negative":[["black","tv"]]},
  {"query":"samsung galaxy phone","positive":[["samsung","phone"]],"negative":[["samsung","shoes"]]},
  {"query":"red laptop","positive":[["red","laptop"]],"negative":[["red","phone"]]},
  {"query":"tablet with stylus","positive":[["stylus","tablet"]],"negative":[["stylus","phone"]]},
  {"query":"smart tv 55 inch","positive":[["55","tv"]],"negative":[["55","phone"]]},
  {"query":"camera with lens","positive":[["lens","camera"]],"negative":[["lens","phone"]]},
  {"query":"headphones with mic","positive":[["mic","headphones"]],"negative":[["mic","laptop"]]},
  {"query":"bluetooth speaker","positive":[["speaker","speaker"]],"negative":[["speaker","phone"]]},
  {"query":"running shoes blue","positive":[["blue","shoes"]],"negative":[["blue","phone"]]},
  {"query":"iphone charger cable","positive":[["charger","iphone"],["cable","iphone"]],"negative":[["charger","tv"]]},
  {"query":"android phone 64gb","positive":[["64gb","phone"]],"negative":[["64gb","laptop"]]},
  {"query":"gaming mouse wireless","positive":[["wireless","mouse"]],"negative":[["wireless","phone"]]},
  {"query":"office chair black","positive":[["black","chair"]],"negative":[["black","phone"]]},
  {"query":"dslr camera 24mp","positive":[["24mp","camera"]],"negative":[["24mp","phone"]]},
  {"query":"smartwatch black strap","positive":[["black","smartwatch"],["strap","smartwatch"]],"negative":[["strap","phone"]]},
  {"query":"usb cable fast charging","positive":[["cable","usb"],["charging","cable"]],"negative":[["cable","tv"]]},
  {"query":"monitor 27 inch","positive":[["27","monitor"]],"negative":[["27","phone"]]},
  {"query":"earbuds with noise cancellation","positive":[["noise","earbuds"]],"negative":[["noise","laptop"]]}
]